# Machine Learning for Applied Research

## 10: Application Challenges in Engineering Contexts

In this notebook, we will cover:
- Small datasets: regularization and learning curves
- Noise and measurement uncertainty: the irreducible error floor
- Imbalance, revisited: class weighting as an alternative to stratification alone
- Interpretability and model validation: permutation importance and partial dependence

### Introduction

Small sample sizes, noisy measurements, class imbalance... We have brushed up against all of these already this week, mostly working around them rather than addressing them directly. This notebook tackles each head-on, with concrete techniques rather than implicit workarounds.

### Setup: Reload Both Datasets


In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Energy Efficiency (regression)
energy = fetch_ucirepo(id=242)
X_e = energy.data.features
y_e = energy.data.targets

X_e.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y_e.columns = ["Heating Load", "Cooling Load"]

target_col_e = y_e.columns[0]

df_e = pd.concat([X_e, y_e[[target_col_e]]], axis=1)
X_e_train, X_e_test, y_e_train, y_e_test = train_test_split(
    df_e.drop(columns=[target_col_e]), df_e[target_col_e], test_size=0.25, random_state=42
)

# AI4I Predictive Maintenance (classification)
maintenance = fetch_ucirepo(id=601)
maint_feature_cols = ["Type", "Air temperature", "Process temperature",
                       "Rotational speed", "Torque", "Tool wear"]
maint_df = pd.concat(
    [maintenance.data.features[maint_feature_cols], maintenance.data.targets[["Machine failure"]]],
    axis=1
)
X_maint = maint_df.drop(columns=["Machine failure"])
y_maint = maint_df["Machine failure"]
X_maint_train, X_maint_test, y_maint_train, y_maint_test = train_test_split(
    X_maint, y_maint, test_size=0.25, random_state=42, stratify=y_maint
)

print(f"Energy Efficiency - Train: {X_e_train.shape[0]} | Test: {X_e_test.shape[0]}")
print(f"AI4I Predictive Maintenance - Train: {X_maint_train.shape[0]} | Test: {X_maint_test.shape[0]}")

### Small Datasets: Regularization and Learning Curves

At 768 rows, Energy Efficiency is small by machine learning standards and several of its features are strongly correlated with each other (Relative Compactness and Surface Area, for instance, move almost perfectly opposite each other). With a small, correlated feature set, plain linear regression can fit noise more easily than you might expect, even without deliberately adding polynomial terms.

**Ridge regression** addresses this by adding a penalty on the size of the coefficients to the least-squares objective, which shrinks them toward zero. This trades a small amount of bias for a meaningful reduction in variance (usually a good trade when data is scarce).

#### 1.1 Exercises:
> (a) Import `Ridge` and `LinearRegression` from the `linear_model` submodule.
>
> (b) Calculate the `cross_val_score` with 5 folds for both the `LinearRegression()` and `Ridge(alpha=10.0)` on on `X_e_train`/`y_e_train`, using the `neg_root_mean_squared_error` scoring.
>
> (c) Print the `.mean()` CV RMSE for both models.
>
> (d) Loop over `alpha` values `[0.1, 1, 10, 50, 100]` for `Ridge`, and for each, compute mean 5-fold CV RMSE (same `scoring` as above).
>
> (e) Which `alpha` gives the lowest CV RMSE?
>
> (f) What would you expect to happen at the extremes, a very small `alpha` (close to 0) and a very large one (say, 10000)? You don't need to run it, reason from what the penalty term does.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.linear_model import Ridge, LinearRegression

plain_scores = cross_val_score(LinearRegression(), X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error")
ridge_scores = cross_val_score(Ridge(alpha=10.0), X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error")

print(f"Plain Linear Regression - Mean CV RMSE: {-plain_scores.mean():.2f}")
print(f"Ridge (alpha=10.0)      - Mean CV RMSE: {-ridge_scores.mean():.2f}")

for alpha in [0.1, 1, 10, 50, 100]:
    scores = cross_val_score(Ridge(alpha=alpha), X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error")
    print(f"alpha={alpha:6.1f} - Mean CV RMSE: {-scores.mean():.2f}")

# (f) As alpha approaches 0, Ridge approaches plain LinearRegression:
# the penalty barely shrinks anything, so it behaves like the
# unregularized model from earlier in this notebook. As alpha grows very
# large, the penalty dominates the objective and coefficients get
# crushed toward zero regardless of what the data says, eventually
# underfitting badly, in the extreme, the model would predict close to
# a constant (the mean of y) regardless of the input features.

#### Learning Curve

A second small-data question: would collecting more data actually help? A **learning curve** answers this by retraining the model on increasingly large subsets of the training data and tracking both training and validation performance at each size.

The `learning_curve` function from the `model_selection` submodule provides the `train_sizes`, `train_scores`, and `val_scores`.

#### 1.2 Exercises:
> (a) Import `learning_curve`.
>
> (b) Track the training and validation performance of a `RandomForestRegressor` with 200 trees and `max_depth=6` on `X_e_train, y_e_train`. Use 5 folds and `train_sizes=np.linspace(0.1, 1.0, 10)`.
>
> (c) Compute `train_rmse_mean = -train_scores.mean(axis=1)` and `val_rmse_mean = -val_scores.mean(axis=1)`.
>
> (d) Plot both against `train_sizes` on a single chart.
>
> (e) Based on the shape of the validation curve, would collecting more building designs likely improve this model meaningfully, or has performance already plateaued? Justify your answer from the plot.

In [ ]:
# Your solution:




##### Solution:

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), random_state=42
)

train_rmse_mean = -train_scores.mean(axis=1)
val_rmse_mean = -val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_rmse_mean, marker="o", label="Train RMSE")
plt.plot(train_sizes, val_rmse_mean, marker="o", label="Validation RMSE")
plt.xlabel("Training examples used")
plt.ylabel("RMSE")
plt.title("Learning curve: Random Forest on Energy Efficiency")
plt.legend()
plt.show()

# (c) If the validation curve is still visibly decreasing at the largest
# training sizes tried, more data would likely help. If it has flattened
# out well before the full training set is used, the model has likely
# already learned what it can from this kind of data, and more of the
# same would not move the needle much, improvement would more likely
# come from better features, a different model, or reducing measurement
# noise instead (the subject of the next section).

### Noise and Measurement Uncertainty

Recall the bias-variance decomposition from Notebook 3: total expected error ≈ bias² + variance + irreducible noise. That last term is not something any model, however well chosen or tuned, can remove. It reflects real randomness or measurement error already present in the data. Energy Efficiency and AI4I are both simulation-derived, so their noise is relatively controlled; real experimental engineering data often has substantially more.

If you have repeated measurements of the same underlying condition, their variance gives you a direct estimate of this noise floor. Without repeats, the following exercise illustrates the same idea in a controlled way: by injecting a known amount of synthetic noise, we can watch the RMSE floor rise in lockstep, even though nothing about the model or features has changed.

#### 1.3 Exercises:
> (a) For noise levels `[0, 1, 3, 5]` (a standard deviation added to the target, in heating-load units), create a noisy training and test target using `np.random.default_rng(42).normal(0, noise_std, size=len(y_e_train))` (and the corresponding size for the test set).
>
> (b) For each noise level, fit a fresh `RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)` on the noisy training target, and record test RMSE against the correspondingly noisy test target.
>
> (c) Does test RMSE increase roughly in line with the injected noise? What does this suggest about interpreting a "disappointing" RMSE on real experimental data, is it necessarily the model's fault?

In [ ]:
# Your solution:






##### Solution:


In [ ]:
noise_levels = [0, 1, 3, 5]
noisy_results = []

for noise_std in noise_levels:
    rng = np.random.default_rng(42)
    y_noisy_train = y_e_train + rng.normal(0, noise_std, size=len(y_e_train))
    y_noisy_test = y_e_test + rng.normal(0, noise_std, size=len(y_e_test))

    noisy_model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
    noisy_model.fit(X_e_train, y_noisy_train)
    noisy_results.append(rmse(y_noisy_test, noisy_model.predict(X_e_test)))

for noise_std, result in zip(noise_levels, noisy_results):
    print(f"Injected noise std={noise_std:4.1f} - Test RMSE: {result:.2f}")

# (c) Test RMSE rises as more noise is injected, even though the model,
# features, and true underlying relationship are all completely
# unchanged. This is the irreducible-error term made directly visible.
# On real experimental data, a "disappointing" RMSE is not automatically
# a sign of a bad model or missing features; it may partly or entirely
# reflect measurement noise inherent in how the data was collected, which
# no amount of modeling effort can remove.

#### 1.4 Exercises:
> (a) At which of the noise levels tried in Exercise 1.3 does the injected noise's standard deviation first exceed the *original* (noise_std=0) model's test RMSE?
>
> (b) Once injected noise is comparable to or larger than your model's baseline error, is further improving the model still the most useful place to invest effort? What would you invest in instead?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
baseline_rmse = noisy_results[0]   # noise_std=0
print(f"Baseline (no injected noise) RMSE: {baseline_rmse:.2f}")
for noise_std, result in zip(noise_levels, noisy_results):
    print(f"noise_std={noise_std} vs. baseline RMSE {baseline_rmse:.2f}")

# (b) Once measurement noise is comparable to or larger than what the
# model already achieves, further model improvements (more tuning, a
# fancier algorithm) will run into a hard floor set by the data itself,
# not the model. At that point, effort is often better spent improving
# the measurement process (better instrumentation, more repeats per
# specimen, tighter experimental control) than on further modeling.

### Imbalance, Revisited: Class Weighting

Notebook 6 and 7 handled AI4I's ~3% failure rate through stratified splitting and threshold-independent metrics like ROC AUC. A complementary technique changes the training process itself: `class_weight="balanced"` tells the model to treat mistakes on the minority class as more costly than mistakes on the majority class, without duplicating or removing any rows. (Resampling techniques, oversampling the minority class or undersampling the majority, are another common approach, though outside what we can cover in three days.)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = ["Type"]
numeric_features = ["Air temperature", "Process temperature",
                     "Rotational speed", "Torque", "Tool wear"]

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features),
])

pipeline_default = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])
pipeline_balanced = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42)),
])

pipeline_default.fit(X_maint_train, y_maint_train)
pipeline_balanced.fit(X_maint_train, y_maint_train)

#### 1.5 Exercises:
> (a) Import `recall_score` and `precision_score` from `sklearn.metrics`. For both `pipeline_default` and `pipeline_balanced`, compute recall and precision for the failure class on the test set.
>
> (b) Which pipeline has higher recall? Which has higher precision?
>
> (c) `class_weight="balanced"` changes this trade-off without changing the training data itself - no rows added or removed. In one or two sentences, explain roughly how it achieves that (hint: think about how the tree-building process uses a per-class weight when scoring a split).

In [ ]:
# Your solution:





##### Solution:

In [ ]:
from sklearn.metrics import recall_score, precision_score

for name, pipe in [("Default", pipeline_default), ("Balanced", pipeline_balanced)]:
    preds = pipe.predict(X_maint_test)
    print(f"{name:10s} - Recall: {recall_score(y_maint_test, preds):.2f} | Precision: {precision_score(y_maint_test, preds):.2f}")

# (c) class_weight="balanced" assigns a higher weight to the minority
# class when computing the criterion used to choose each split, and when
# scoring how well a leaf is doing overall. The training process is
# effectively told that misclassifying a failure counts for much more
# than misclassifying a normal case, even though every row is still used
# exactly as it appeared in the data, no resampling or duplication
# involved.

### Interpretability and Model Validation

Two tools help you sanity-check what a model actually learned, rather than trusting a good test score blindly:

- **Permutation importance**: shuffle one feature's values at a time (breaking its relationship with the target) and measure how much performance drops. Unlike a tree's built-in `feature_importances_`, this works for any model and is computed on held-out data, which makes it more trustworthy.
- **Partial dependence**: shows how the model's average prediction changes as one feature varies, holding the others at their observed distribution: a direct way to check whether the model learned a physically sensible relationship.

#### 1.6 Exercises:
> (a) Train another `RandomForestRegressor()` (200 trees, depth 6), initialized as `forest_e`.
>
> (b) Import `PartialDependenceDisplay` from `sklearn.inspection`.
>
> (c) Plot the partial dependence of `forest_e`'s predictions on `Overall Height` using `PartialDependenceDisplay.from_estimator(forest_e, X_e_train, features=["Overall Height"])`.
>
> (d) Does the direction of the relationship (does predicted heating load rise or fall as height increases?) match what you found via correlation back in Notebook 1? Would a partial dependence plot showing the *opposite* direction necessarily be a red flag, or could there be a legitimate reason for it?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

forest_e = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
forest_e.fit(X_e_train, y_e_train)

PartialDependenceDisplay.from_estimator(forest_e, X_e_train, features=["Overall Height"])
plt.show()

# (c) The curve should rise with Overall Height, matching the strong
# positive correlation (around 0.89) found back in Notebook 1: taller
# buildings, for a similar footprint, need more energy to heat. If a
# model showed the opposite direction, that would usually be a red flag:
# a data problem (a leaked or miscoded feature), or in the worst case, 
# a bug in how the features were prepared. 
# It could occasionally reflect an interaction effect that only appears
# when other features are held at unusual values, but that possibility
# is worth confirming deliberately, not assuming by default. A check
# like this is a cheap, fast sanity test worth running on any model
# before trusting its conclusions.

#### Challenge:
> Compute permutation importance for `pipeline_balanced` on `X_maint_test`/`y_maint_test`, scoring on `"roc_auc"`. Compare the resulting ranking (over the *original* feature columns) to `pipeline_balanced.named_steps["classifier"].feature_importances_` (over the *one-hot encoded* columns, retrievable via `pipeline_balanced.named_steps["preprocessor"].get_feature_names_out()`). Do the two feature spaces even match up? Which ranking would you trust more, and why?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.inspection import permutation_importance

perm_result_maint = permutation_importance(
    pipeline_balanced, X_maint_test, y_maint_test, n_repeats=10, random_state=42, scoring="roc_auc"
)

perm_importance_df = pd.DataFrame({
    "feature": X_maint_test.columns,
    "permutation_importance": perm_result_maint.importances_mean,
}).sort_values("permutation_importance", ascending=False)

encoded_feature_names = pipeline_balanced.named_steps["preprocessor"].get_feature_names_out()
builtin_importance_df = pd.DataFrame({
    "feature": encoded_feature_names,
    "builtin_importance": pipeline_balanced.named_steps["classifier"].feature_importances_,
}).sort_values("builtin_importance", ascending=False)

print("Permutation importance (original features):")
print(perm_importance_df)
print("\nBuilt-in feature_importances_ (one-hot encoded features):")
print(builtin_importance_df)

# The two rankings are not even over the same features: permutation
# importance operates on the original columns, so Type is scored as one
# feature, while feature_importances_ is computed after Type has been
# split into separate one-hot columns, fragmenting its importance across
# cat__Type_H, cat__Type_L, and cat__Type_M individually. This is itself
# a good reason to prefer permutation importance when you care about the
# original, human-meaningful variables: it naturally respects your
# feature engineering choices instead of splintering them.

## Conclusion

This notebook addressed the conditions that define most applied engineering ML work, rather than treating them as afterthoughts:

- **Small datasets**: regularization (Ridge) trades a little bias for meaningfully less variance, with `alpha` controlling exactly how much; learning curves tell you whether more data would actually help
- **Noise and measurement uncertainty**: the irreducible error floor is not a modeling failure buts a property of the data itself, and once injected noise rivals your model's baseline error, further modeling effort runs into diminishing returns
- **Imbalance, revisited**: `class_weight="balanced"` reshapes the training objective directly, complementing (not replacing) stratification and threshold-independent metrics
- **Interpretability**: permutation importance and partial dependence plots let you sanity-check a model's learned relationships against domain knowledge, rather than trusting a good score blindly